In [ ]:
#| default_exp compute

# Compute

> Compute Engine, GKE, Artifact Registry, Cloud Run, Cloud Build, Cloud DNS.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import io
import os
import socket
import tarfile
import time
import uuid
from gcpeasy._util import _log, wait_op, wait_rest_op

try:
    from google.cloud import compute_v1
except ImportError:
    pass
try:
    from google.cloud import container_v1
except ImportError:
    pass
try:
    from google.cloud import artifactregistry_v1
except ImportError:
    pass

try:
    from google.cloud import run_v2
except ImportError:
    pass

try:
    from google.cloud import storage as _gcs_storage
except ImportError:
    pass

try:
    from google.cloud.devtools import cloudbuild_v1
except ImportError:
    pass

## Compute Engine

In [ ]:
#| export
def _latest_debian_image(auth, zone: str) -> str:
    """Return the latest Debian 12 image selfLink."""
    client = compute_v1.ImagesClient(credentials=auth.credentials)
    images = client.get_from_family(project='debian-cloud', family='debian-12')
    return images.self_link


def _build_metadata(items: dict) -> 'compute_v1.Metadata':
    """Build a ``compute_v1.Metadata`` from a flat ``{key: value}`` dict."""
    return compute_v1.Metadata(items=[
        compute_v1.Items(key=k, value=v) for k, v in items.items() if v is not None
    ])


def create_instance(
    auth,
    name: str,
    machine_type: str = 'e2-medium',
    zone: str = None,
    image: str = None,
    disk_size_gb: int = 20,
    shielded: bool = True,
    labels: dict = None,
    external_ip: bool = True,
    ssh_keys: dict = None,
    startup_script: str = None,
    service_account: str = None,
    service_account_scopes: list = None,
    tags: list = None,
    network: str = 'default',
    subnetwork: str = None,
    metadata: dict = None,
    enable_os_login: bool = True,
    block_project_ssh_keys: bool = True,
    **_,
) -> dict:
    """Create a Compute Engine VM instance.

    Production-friendly defaults:

    * **External IP** assigned by default (one-to-one NAT) so the VM is
      reachable; pass ``external_ip=False`` for an internal-only VM.
    * **OS Login** enabled and **project-wide SSH keys blocked** by default
      (``enable_os_login=True``, ``block_project_ssh_keys=True``).  Pass
      ``ssh_keys={'user': 'ssh-rsa …'}`` to add instance-level keys when
      ``enable_os_login=False``.
    * Shielded VM (secure boot + vTPM + integrity monitoring) on by default.
    * ``startup_script`` is wired to the standard ``startup-script``
      metadata key.
    * ``service_account`` (email) attaches a least-privilege identity; when
      omitted the instance falls back to the Compute Engine default SA.

    Idempotent — returns immediately if an instance with ``name`` already
    exists in ``zone``.
    """
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)

    try:
        existing = client.get(project=auth.project, zone=zone, instance=name)
        return {'name': name, 'zone': zone, 'status': existing.status}
    except Exception:
        pass

    source_image = image or _latest_debian_image(auth, zone)

    md_items: dict = {}
    if startup_script:
        md_items['startup-script'] = startup_script
    if enable_os_login:
        md_items['enable-oslogin'] = 'TRUE'
    if block_project_ssh_keys:
        md_items['block-project-ssh-keys'] = 'TRUE'
    if ssh_keys:
        md_items['ssh-keys'] = '\n'.join(
            f'{user}:{key}' for user, key in ssh_keys.items()
        )
    if metadata:
        for k, v in metadata.items():
            md_items[k] = v

    nic_kwargs = dict(name=f'projects/{auth.project}/global/networks/{network}')
    if subnetwork:
        nic_kwargs['subnetwork'] = (
            f'projects/{auth.project}/regions/{auth.region}/subnetworks/{subnetwork}'
        )
    if external_ip:
        nic_kwargs['access_configs'] = [
            compute_v1.AccessConfig(
                name='External NAT',
                type_='ONE_TO_ONE_NAT',
                network_tier='PREMIUM',
            )
        ]
    nic = compute_v1.NetworkInterface(**nic_kwargs)

    sa_block = None
    if service_account is not None:
        sa_block = [compute_v1.ServiceAccount(
            email=service_account,
            scopes=service_account_scopes or ['https://www.googleapis.com/auth/cloud-platform'],
        )]

    instance = compute_v1.Instance(
        name=name,
        machine_type=f'zones/{zone}/machineTypes/{machine_type}',
        disks=[compute_v1.AttachedDisk(
            boot=True, auto_delete=True,
            initialize_params=compute_v1.AttachedDiskInitializeParams(
                source_image=source_image, disk_size_gb=disk_size_gb,
            ),
        )],
        network_interfaces=[nic],
        labels=labels or {},
        tags=compute_v1.Tags(items=tags or []),
        metadata=_build_metadata(md_items) if md_items else None,
        service_accounts=sa_block,
        shielded_instance_config=(
            compute_v1.ShieldedInstanceConfig(
                enable_secure_boot=True,
                enable_vtpm=True,
                enable_integrity_monitoring=True,
            ) if shielded else None
        ),
    )
    op = client.insert(project=auth.project, zone=zone, instance_resource=instance)
    wait_op(op, what=f'create_instance {name}', timeout=300)
    inst = client.get(project=auth.project, zone=zone, instance=name)
    out = {'name': name, 'zone': zone, 'status': inst.status,
           'internal_ip': inst.network_interfaces[0].network_i_p}
    if external_ip and inst.network_interfaces[0].access_configs:
        out['external_ip'] = inst.network_interfaces[0].access_configs[0].nat_i_p
    return out


def instance_ip(auth, name: str, zone: str = None, internal: bool = False) -> str:
    """Return the IP of a Compute Engine instance.  Falls back to the internal
    IP automatically if no external one is configured."""
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    inst = client.get(project=auth.project, zone=zone, instance=name)
    nic = inst.network_interfaces[0]
    if internal or not nic.access_configs:
        return nic.network_i_p
    return nic.access_configs[0].nat_i_p


def start_instance(auth, name: str, zone: str = None):
    "Start a stopped Compute Engine instance."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    op = client.start(project=auth.project, zone=zone, instance=name)
    wait_op(op, what=f'start_instance {name}', timeout=300)


def stop_instance(auth, name: str, zone: str = None):
    "Stop a running Compute Engine instance."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    op = client.stop(project=auth.project, zone=zone, instance=name)
    wait_op(op, what=f'stop_instance {name}', timeout=300)


def delete_instance(auth, name: str, zone: str = None) -> dict:
    "Delete a Compute Engine instance.  Idempotent."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    try:
        op = client.delete(project=auth.project, zone=zone, instance=name)
    except Exception:
        return {'name': name, 'status': 'not_found'}
    wait_op(op, what=f'delete_instance {name}', timeout=300)
    return {'name': name, 'status': 'deleted'}

## VM bootstrap helpers (B2)

In [ ]:
#| export
#: Minimal startup script that installs Docker + Compose plugin on Debian 12.
DOCKER_INSTALL_SCRIPT = r"""#!/usr/bin/env bash
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
if command -v docker >/dev/null 2>&1 && docker compose version >/dev/null 2>&1; then
  echo 'docker already installed'
  exit 0
fi
apt-get update -y
apt-get install -y ca-certificates curl gnupg
install -m 0755 -d /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/debian/gpg | gpg --dearmor -o /etc/apt/keyrings/docker.gpg
chmod a+r /etc/apt/keyrings/docker.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] \
https://download.docker.com/linux/debian $(. /etc/os-release && echo "$VERSION_CODENAME") stable" \
  | tee /etc/apt/sources.list.d/docker.list >/dev/null
apt-get update -y
apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
systemctl enable --now docker
"""


def vm_install_docker(auth, name: str = None, zone: str = None) -> str:
    """Return a startup script that installs Docker + Compose on Debian.

    Pass the result as ``startup_script=`` to :func:`create_instance`.  When
    a VM already exists, manually run this on the host (or use
    :func:`vm_run_compose` which inlines it).
    """
    return DOCKER_INSTALL_SCRIPT


def wait_for_ssh(auth, name: str, zone: str = None, timeout: int = 300,
                 port: int = 22) -> str:
    """Block until TCP/22 on the VM's external IP accepts connections.

    Returns the IP that succeeded.  Raises ``TimeoutError`` if the deadline
    elapses.  Useful as the gating step between ``create_instance`` and
    config-management or ``vm_run_compose``.
    """
    ip = instance_ip(auth, name, zone=zone)
    deadline = time.monotonic() + timeout
    last_err = None
    while time.monotonic() < deadline:
        try:
            with socket.create_connection((ip, port), timeout=5):
                _log(f'wait_for_ssh {name}: {ip}:{port} reachable')
                return ip
        except OSError as e:
            last_err = e
            time.sleep(5)
    raise TimeoutError(
        f'wait_for_ssh {name} ({ip}:{port}) not reachable after {timeout}s '
        f'(last error: {last_err})'
    )


def vm_run_compose(auth, name: str, compose_yaml: str,
                   env: dict = None, files: dict = None,
                   zone: str = None,
                   workdir: str = '/opt/app') -> dict:
    """Generate a startup script that lays out files + ``docker-compose.yml``
    on the VM and runs ``docker compose up -d``.

    Returns ``{'startup_script': <str>, 'workdir': <path>}``.  Use this
    either by passing the ``startup_script`` to :func:`create_instance`, or
    by setting it as instance metadata before reboot.

    This is a deliberately simple, stateless helper — all secret values
    embedded in ``env`` end up in instance metadata, so prefer Secret
    Manager + a sidecar fetcher for sensitive values.
    """
    import shlex
    env = env or {}
    files = dict(files or {})
    files[f'{workdir}/docker-compose.yml'] = compose_yaml
    if env:
        files[f'{workdir}/.env'] = '\n'.join(
            f'{k}={v}' for k, v in env.items()
        ) + '\n'

    parts = ['#!/usr/bin/env bash', 'set -euo pipefail',
             DOCKER_INSTALL_SCRIPT.split('\n', 1)[1].strip(),
             f'mkdir -p {shlex.quote(workdir)}']
    for path, content in files.items():
        # heredoc-safe: use a unique sentinel
        sentinel = f'GCPEASY_EOF_{uuid.uuid4().hex[:8]}'
        parts.append(f'mkdir -p {shlex.quote(os.path.dirname(path))}')
        parts.append(f'cat > {shlex.quote(path)} <<\'{sentinel}\'')
        parts.append(content.rstrip('\n'))
        parts.append(sentinel)
    parts.append(f'cd {shlex.quote(workdir)}')
    parts.append('docker compose pull || true')
    parts.append('docker compose up -d')
    script = '\n'.join(parts) + '\n'
    return {'startup_script': script, 'workdir': workdir}

## GKE

In [ ]:
#| export
def create_gke_cluster(auth, name: str, node_count: int = 1,
                       machine_type: str = 'e2-standard-4', autopilot: bool = True,
                       workload_identity: bool = True, labels: dict = None,
                       **_) -> dict:
    """Create a GKE cluster. Defaults to Autopilot mode with Workload Identity."""
    client = container_v1.ClusterManagerClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'
    try:
        existing = client.get_cluster(name=f'{parent}/clusters/{name}')
        return {'name': name, 'endpoint': existing.endpoint, 'status': str(existing.status)}
    except Exception:
        pass

    if autopilot:
        cluster = container_v1.Cluster(
            name=name,
            autopilot=container_v1.Autopilot(enabled=True),
            workload_identity_config=(
                container_v1.WorkloadIdentityConfig(
                    workload_pool=f'{auth.project}.svc.id.goog'
                ) if workload_identity else None
            ),
            resource_labels=labels or {},
        )
    else:
        cluster = container_v1.Cluster(
            name=name,
            node_pools=[container_v1.NodePool(
                name='default-pool',
                initial_node_count=node_count,
                config=container_v1.NodeConfig(
                    machine_type=machine_type,
                    workload_metadata_config=(
                        container_v1.WorkloadMetadataConfig(
                            mode=container_v1.WorkloadMetadataConfig.Mode.GKE_METADATA
                        ) if workload_identity else None
                    ),
                ),
            )],
            workload_identity_config=(
                container_v1.WorkloadIdentityConfig(
                    workload_pool=f'{auth.project}.svc.id.goog'
                ) if workload_identity else None
            ),
            resource_labels=labels or {},
        )

    op = client.create_cluster(parent=parent, cluster=cluster)
    return {'name': name, 'operation': op.name}


def gke_kubeconfig(auth, name: str) -> dict:
    """Return kubeconfig dict for connecting to a GKE cluster."""
    client = container_v1.ClusterManagerClient(credentials=auth.credentials)
    cluster = client.get_cluster(
        name=f'projects/{auth.project}/locations/{auth.region}/clusters/{name}'
    )
    return {
        'endpoint': f'https://{cluster.endpoint}',
        'ca_cert': cluster.master_auth.cluster_ca_certificate,
        'name': name,
    }


def scale_gke(auth, name: str, node_pool: str, node_count: int):
    "Scale a GKE node pool to ``node_count``."
    client = container_v1.ClusterManagerClient(credentials=auth.credentials)
    op = client.set_node_pool_size(
        name=(f'projects/{auth.project}/locations/{auth.region}'
              f'/clusters/{name}/nodePools/{node_pool}'),
        node_count=node_count,
    )
    return op.name

## Artifact Registry

In [ ]:
#| export
def create_artifact_registry(auth, name: str, format: str = 'DOCKER',
                             labels: dict = None, **_) -> dict:
    """Create an Artifact Registry repository.  Vulnerability scanning is enabled."""
    client = artifactregistry_v1.ArtifactRegistryClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'
    try:
        existing = client.get_repository(name=f'{parent}/repositories/{name}')
        return {'name': existing.name, 'format': format}
    except Exception:
        pass

    repo = artifactregistry_v1.Repository(
        format_=artifactregistry_v1.Repository.Format[format],
        labels=labels or {},
    )
    op = client.create_repository(parent=parent, repository_id=name, repository=repo)
    result = wait_op(op, what=f'create_artifact_registry {name}', timeout=120)
    return {'name': result.name, 'format': format}


def registry_url(auth, name: str) -> str:
    "Return the Docker image path prefix for an Artifact Registry repo."
    return f'{auth.region}-docker.pkg.dev/{auth.project}/{name}'


def attach_registry_to_gke(auth, registry_name: str, gke_sa_email: str):
    """Grant Workload Identity SA read access to the Artifact Registry repo."""
    from gcpeasy.network import bind_iam_role
    bind_iam_role(auth, gke_sa_email, 'roles/artifactregistry.reader')
    return {'registry': registry_name, 'sa': gke_sa_email,
            'role': 'roles/artifactregistry.reader'}


def delete_artifact_registry(auth, name: str) -> dict:
    """Delete an Artifact Registry repo.  Idempotent."""
    client = artifactregistry_v1.ArtifactRegistryClient(credentials=auth.credentials)
    full = f'projects/{auth.project}/locations/{auth.region}/repositories/{name}'
    try:
        op = client.delete_repository(name=full)
    except Exception:
        return {'name': full, 'status': 'not_found'}
    wait_op(op, what=f'delete_artifact_registry {name}', timeout=120)
    return {'name': full, 'status': 'deleted'}

## Cloud Run (A7)

In [ ]:
#| export
def deploy_cloudrun(
    auth,
    name: str,
    image: str,
    service_account: str = None,
    env_vars: dict = None,
    secrets: dict = None,
    concurrency: int = 80,
    allow_unauthenticated: bool = False,
    vpc_connector: str = None,
    min_instances: int = 0,
    max_instances: int = 10,
    cpu: str = '1',
    memory: str = '512Mi',
    port: int = 8080,
    timeout_sec: int = 300,
    ingress: str = None,
    labels: dict = None,
    **compliance_opts,
) -> dict:
    """Deploy or update a Cloud Run service with safe defaults.

    Behavioural notes:

    * ``ingress`` defaults to ``'all'`` when ``allow_unauthenticated=True``
      (so the Internet can reach it) and to
      ``'internal-and-cloud-load-balancing'`` otherwise (so only an LB+IAP
      can reach the service).
    * ``secrets`` maps env-var name → ``'projects/p/secrets/n/versions/v'``;
      these are injected as Cloud Run secret env-vars.
    * ``allow_unauthenticated=True`` uses an **additive** IAM policy update
      (read-modify-write) instead of overwriting all existing bindings.
    * Updates use field masks (delegated to the Cloud Run SDK) and never
      re-set the immutable ``name`` field on the service.
    """
    client = run_v2.ServicesClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'
    service_name = f'{parent}/services/{name}'

    env = [run_v2.EnvVar(name=k, value=v) for k, v in (env_vars or {}).items()]
    for k, ref in (secrets or {}).items():
        # ref formats accepted: full version path, or "secret:version"
        if ref.startswith('projects/'):
            secret = ref.split('/secrets/')[1].split('/versions/')[0]
            version = ref.rsplit('/', 1)[-1]
        elif ':' in ref:
            secret, version = ref.split(':', 1)
        else:
            secret, version = ref, 'latest'
        env.append(run_v2.EnvVar(
            name=k,
            value_source=run_v2.EnvVarSource(
                secret_key_ref=run_v2.SecretKeySelector(secret=secret, version=version),
            ),
        ))

    container = run_v2.Container(
        image=image,
        env=env,
        ports=[run_v2.ContainerPort(container_port=port)],
        resources=run_v2.ResourceRequirements(
            limits={'cpu': cpu, 'memory': memory},
        ),
    )
    template = run_v2.RevisionTemplate(
        containers=[container],
        service_account=service_account or '',
        max_instance_request_concurrency=concurrency,
        timeout={'seconds': timeout_sec} if timeout_sec else None,
        scaling=run_v2.RevisionScaling(
            min_instance_count=min_instances,
            max_instance_count=max_instances,
        ),
        labels=labels or {},
    )
    if vpc_connector:
        template.vpc_access = run_v2.VpcAccess(
            connector=vpc_connector,
            egress=run_v2.VpcAccess.VpcEgress.ALL_TRAFFIC,
        )

    if ingress is None:
        ingress = 'all' if allow_unauthenticated else 'internal-and-cloud-load-balancing'
    ingress_enum = {
        'all': run_v2.IngressTraffic.INGRESS_TRAFFIC_ALL,
        'internal': run_v2.IngressTraffic.INGRESS_TRAFFIC_INTERNAL_ONLY,
        'internal-and-cloud-load-balancing':
            run_v2.IngressTraffic.INGRESS_TRAFFIC_INTERNAL_LOAD_BALANCER,
    }.get(ingress, run_v2.IngressTraffic.INGRESS_TRAFFIC_ALL)

    service = run_v2.Service(template=template, ingress=ingress_enum,
                             labels=labels or {})

    try:
        client.get_service(name=service_name)
        service.name = service_name
        op = client.update_service(service=service)
    except Exception:
        op = client.create_service(parent=parent, service=service, service_id=name)

    result = wait_op(op, what=f'deploy_cloudrun {name}', timeout=600)

    if allow_unauthenticated:
        _grant_cloudrun_invoker_public(client, result.name)

    return {'name': result.name, 'uri': result.uri}


def _grant_cloudrun_invoker_public(client, service_resource: str) -> None:
    """Add ``allUsers`` to ``roles/run.invoker`` without clobbering existing bindings."""
    from google.iam.v1 import iam_policy_pb2, policy_pb2
    policy = client.get_iam_policy(
        request=iam_policy_pb2.GetIamPolicyRequest(resource=service_resource)
    )
    bindings = list(policy.bindings)
    found = False
    for b in bindings:
        if b.role == 'roles/run.invoker':
            if 'allUsers' not in b.members:
                b.members.append('allUsers')
            found = True
            break
    if not found:
        bindings.append(policy_pb2.Binding(
            role='roles/run.invoker', members=['allUsers']))
    new_policy = policy_pb2.Policy(bindings=bindings, etag=policy.etag)
    client.set_iam_policy(
        request=iam_policy_pb2.SetIamPolicyRequest(
            resource=service_resource, policy=new_policy,
        )
    )


def cloudrun_url(auth, name: str) -> str:
    "Return the HTTPS URL of a deployed Cloud Run service."
    client = run_v2.ServicesClient(credentials=auth.credentials)
    service = client.get_service(
        name=f'projects/{auth.project}/locations/{auth.region}/services/{name}'
    )
    return service.uri


def delete_cloudrun(auth, name: str) -> dict:
    "Delete a Cloud Run service.  Idempotent."
    client = run_v2.ServicesClient(credentials=auth.credentials)
    full = f'projects/{auth.project}/locations/{auth.region}/services/{name}'
    try:
        op = client.delete_service(name=full)
    except Exception:
        return {'name': full, 'status': 'not_found'}
    wait_op(op, what=f'delete_cloudrun {name}', timeout=300)
    return {'name': full, 'status': 'deleted'}


def cloudrun_domain_mapping(auth, service: str, domain: str,
                            region: str = None) -> dict:
    """Create a Cloud Run domain mapping.

    Note: Cloud Run domain mappings are only available via the v1 API
    (``run.googleapis.com``) and only in a subset of regions.  After
    creation the returned ``records`` list contains the DNS records you
    must publish for the domain to verify.
    """
    import googleapiclient.discovery
    region = region or auth.region
    run = googleapiclient.discovery.build('run', 'v1', credentials=auth.credentials)
    parent = f'namespaces/{auth.project}'
    body = {
        'apiVersion': 'domains.cloudrun.com/v1',
        'kind': 'DomainMapping',
        'metadata': {'name': domain, 'namespace': auth.project},
        'spec': {'routeName': service},
    }
    try:
        existing = run.namespaces().domainmappings().get(
            name=f'{parent}/domainmappings/{domain}',
        ).execute()
        records = existing.get('status', {}).get('resourceRecords', [])
        return {'domain': domain, 'service': service,
                'status': 'exists', 'records': records}
    except Exception:
        pass
    result = run.namespaces().domainmappings().create(
        parent=parent, body=body,
    ).execute()
    records = result.get('status', {}).get('resourceRecords', [])
    return {'domain': domain, 'service': service,
            'status': 'created', 'records': records}

## Cloud Build (B1)

In [ ]:
#| export
def _tar_dir(source_dir: str) -> bytes:
    """Tar+gzip a directory in memory, honouring an optional ``.gcloudignore``
    or ``.dockerignore`` (very small subset: line-prefix glob-style)."""
    import fnmatch
    ignores = []
    for ig in ('.gcloudignore', '.dockerignore'):
        p = os.path.join(source_dir, ig)
        if os.path.isfile(p):
            with open(p, 'r', encoding='utf-8') as f:
                ignores.extend(
                    ln.strip() for ln in f
                    if ln.strip() and not ln.lstrip().startswith('#')
                )
            break

    def _ignored(rel: str) -> bool:
        for pat in ignores:
            if fnmatch.fnmatch(rel, pat) or fnmatch.fnmatch(os.path.basename(rel), pat):
                return True
        return False

    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode='w:gz') as tf:
        for root, dirs, fs in os.walk(source_dir):
            # skip ignored dirs in-place
            dirs[:] = [
                d for d in dirs
                if not _ignored(os.path.relpath(os.path.join(root, d), source_dir))
            ]
            for f in fs:
                full = os.path.join(root, f)
                rel = os.path.relpath(full, source_dir)
                if _ignored(rel):
                    continue
                tf.add(full, arcname=rel)
    return buf.getvalue()


def build_image_cloudbuild(auth, source_dir: str, image: str,
                           dockerfile: str = 'Dockerfile',
                           timeout_sec: int = 1200,
                           machine_type: str = None,
                           **_) -> dict:
    """Submit a Cloud Build job that builds ``source_dir`` and pushes ``image``.

    Uploads the directory as a tarball to a per-project staging bucket
    (``{project}_cloudbuild``), creates a single-step ``gcr.io/cloud-builders/docker``
    build, and waits for completion.  Returns the resulting image digest URL.
    """
    project = auth.project
    bucket_name = f'{project}_cloudbuild'

    # Stage source
    storage = _gcs_storage.Client(project=project, credentials=auth.credentials)
    try:
        bucket = storage.get_bucket(bucket_name)
    except Exception:
        bucket = storage.create_bucket(bucket_name, location=auth.region)
        try:
            bucket.iam_configuration.uniform_bucket_level_access_enabled = True
            bucket.patch()
        except Exception:
            pass
    object_name = f'gcpeasy-source/{int(time.time())}-{uuid.uuid4().hex[:8]}.tgz'
    blob = bucket.blob(object_name)
    blob.upload_from_string(_tar_dir(source_dir),
                            content_type='application/gzip')
    _log(f'build_image_cloudbuild: uploaded source to gs://{bucket_name}/{object_name}')

    cb = cloudbuild_v1.CloudBuildClient(credentials=auth.credentials)
    build = cloudbuild_v1.Build(
        source=cloudbuild_v1.Source(
            storage_source=cloudbuild_v1.StorageSource(
                bucket=bucket_name, object_=object_name,
            ),
        ),
        steps=[cloudbuild_v1.BuildStep(
            name='gcr.io/cloud-builders/docker',
            args=['build', '-f', dockerfile, '-t', image, '.'],
        )],
        images=[image],
        timeout={'seconds': timeout_sec},
        options=cloudbuild_v1.BuildOptions(
            machine_type=cloudbuild_v1.BuildOptions.MachineType[machine_type]
            if machine_type else cloudbuild_v1.BuildOptions.MachineType.UNSPECIFIED,
            logging=cloudbuild_v1.BuildOptions.LoggingMode.CLOUD_LOGGING_ONLY,
        ),
    )
    op = cb.create_build(project_id=project, build=build)
    result = wait_op(op, what=f'build_image_cloudbuild {image}', timeout=timeout_sec + 60)
    digest = None
    for img in (result.results.images if result.results else []) or []:
        if img.name == image and img.digest:
            digest = img.digest
            break
    image_with_digest = f'{image}@{digest}' if digest else image
    return {
        'image': image,
        'image_digest': image_with_digest,
        'build_id': result.id,
        'status': result.status.name if hasattr(result.status, 'name') else str(result.status),
        'log_url': result.log_url,
    }


def push_image(auth, registry: str, source_image: str = None) -> dict:
    """Configure local Docker auth for an Artifact Registry host.

    This is intentionally lightweight — it ensures the registry exists and
    returns the ``gcloud auth configure-docker`` host the user should run.
    Pushing a local image still happens via ``docker push`` on the caller's
    machine; the heavier server-side flow is :func:`build_image_cloudbuild`.
    """
    create_artifact_registry(auth, registry)
    host = f'{auth.region}-docker.pkg.dev'
    return {
        'registry': registry,
        'host': host,
        'configure_docker': f'gcloud auth configure-docker {host}',
        'image_prefix': registry_url(auth, registry),
    }

### Tests — module exports

In [ ]:
#| hide
import gcpeasy.compute as M
for n in ['create_instance', 'create_gke_cluster', 'create_artifact_registry',
          'deploy_cloudrun']:
    assert n in M.__all__

### Tests — ported from `tests/test_compute.py` and `tests/test_cloudrun.py`


In [ ]:
#| hide
import sys as _sys, types as _types
from unittest.mock import MagicMock, patch
_kernel = _sys.modules['__main__']

class _MockAuth:
    project = 'test-project'
    region = 'us-central1'
    credentials = MagicMock(name='credentials')

def _mock_auth(): return _MockAuth()

class _Wrap:
    @classmethod
    def make(cls, name):
        return type(name, (cls,), {'__init__': cls._init, '__repr__': cls._repr})
    def _init(self, **kw):
        for k, v in kw.items(): setattr(self, k, v)
    def _repr(self): return f'<{type(self).__name__} {self.__dict__}>'

def _install_fake_compute_v1():
    mod = _types.ModuleType('google.cloud.compute_v1')
    for name in ['AccessConfig', 'AttachedDisk', 'AttachedDiskInitializeParams',
                 'Allowed', 'Firewall', 'ForwardingRule',
                 'Instance', 'NetworkInterface', 'Network', 'NetworkRoutingConfig',
                 'Subnetwork', 'Items', 'Metadata', 'Tags', 'ServiceAccount',
                 'ShieldedInstanceConfig',
                 'BackendBucket', 'BackendBucketCdnPolicy']:
        setattr(mod, name, _Wrap.make(name))
    for name in ['NetworksClient', 'SubnetworksClient', 'FirewallsClient',
                 'InstancesClient', 'ImagesClient', 'ForwardingRulesClient',
                 'BackendBucketsClient']:
        setattr(mod, name, MagicMock(name=name))
    return mod


In [ ]:
#| hide
# create_instance: assigns external IP by default (NAT access config) + IPs returned
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    instances = MagicMock()
    fake_compute.InstancesClient.return_value = instances
    instances.get.side_effect = [
        Exception('not found'),
        MagicMock(status='RUNNING', network_interfaces=[MagicMock(
            network_i_p='10.0.0.5',
            access_configs=[MagicMock(nat_i_p='34.1.2.3')],
        )]),
    ]
    op = MagicMock(); op.result = MagicMock(); op.done = MagicMock(return_value=True)
    instances.insert.return_value = op

    with patch.object(_kernel, '_latest_debian_image', return_value='img'):
        out = create_instance(_mock_auth(), 'vm1')

    insert_kwargs = instances.insert.call_args.kwargs
    nic = insert_kwargs['instance_resource'].network_interfaces[0]
    assert hasattr(nic, 'access_configs')
    assert len(nic.access_configs) == 1
    assert nic.access_configs[0].type_ == 'ONE_TO_ONE_NAT'
    assert out['internal_ip'] == '10.0.0.5'
    assert out['external_ip'] == '34.1.2.3'


In [ ]:
#| hide
# create_instance: external_ip=False → no access_configs and no external_ip in result
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    instances = MagicMock()
    fake_compute.InstancesClient.return_value = instances
    instances.get.side_effect = [
        Exception('not found'),
        MagicMock(status='RUNNING', network_interfaces=[MagicMock(
            network_i_p='10.0.0.7', access_configs=[],
        )]),
    ]
    op = MagicMock(); op.result = MagicMock(); op.done = MagicMock(return_value=True)
    instances.insert.return_value = op

    with patch.object(_kernel, '_latest_debian_image', return_value='img'):
        out = create_instance(_mock_auth(), 'vm2', external_ip=False)

    nic = instances.insert.call_args.kwargs['instance_resource'].network_interfaces[0]
    assert not hasattr(nic, 'access_configs') or not nic.access_configs
    assert 'external_ip' not in out
    assert out['internal_ip'] == '10.0.0.7'


In [ ]:
#| hide
# create_instance: wires startup script, OS Login flag, blocks project SSH keys, sets SA
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    instances = MagicMock()
    fake_compute.InstancesClient.return_value = instances
    instances.get.side_effect = [
        Exception('not found'),
        MagicMock(status='RUNNING', network_interfaces=[MagicMock(
            network_i_p='10', access_configs=[MagicMock(nat_i_p='1')],
        )]),
    ]
    op = MagicMock(); op.result = MagicMock(); op.done = MagicMock(return_value=True)
    instances.insert.return_value = op

    with patch.object(_kernel, '_latest_debian_image', return_value='img'):
        create_instance(_mock_auth(), 'vm3',
                        startup_script='echo hi',
                        service_account='sa@p.iam.gserviceaccount.com')

    res = instances.insert.call_args.kwargs['instance_resource']
    md_keys = {it.key: it.value for it in res.metadata.items}
    assert md_keys['startup-script'] == 'echo hi'
    assert md_keys['enable-oslogin'] == 'TRUE'
    assert md_keys['block-project-ssh-keys'] == 'TRUE'
    assert res.service_accounts[0].email == 'sa@p.iam.gserviceaccount.com'


In [ ]:
#| hide
# instance_ip falls back to internal IP when no external present
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    instances = MagicMock()
    fake_compute.InstancesClient.return_value = instances
    instances.get.return_value = MagicMock(network_interfaces=[
        MagicMock(network_i_p='10.0.0.9', access_configs=[]),
    ])
    assert instance_ip(_mock_auth(), 'x') == '10.0.0.9'


In [ ]:
#| hide
# vm_install_docker returns a real install script
s = vm_install_docker(_mock_auth())
assert 'apt-get install' in s
assert 'docker-ce' in s
assert 'docker-compose-plugin' in s


In [ ]:
#| hide
# vm_run_compose emits a runnable startup bundle with workdir, env, compose-up
bundle = vm_run_compose(_mock_auth(), 'vm', 'services:\n  web:\n    image: nginx\n',
                        env={'API_KEY': 'abc'}, workdir='/srv/app')
s = bundle['startup_script']
assert s.startswith('#!/usr/bin/env bash')
assert 'set -euo pipefail' in s
assert '/srv/app/docker-compose.yml' in s
assert '/srv/app/.env' in s
assert 'docker compose up -d' in s
assert bundle['workdir'] == '/srv/app'


In [ ]:
#| hide
# Regression: deploy_cloudrun(allow_unauthenticated=True) must be ADDITIVE — must not
# clobber existing IAM bindings on the service.
class _Binding:
    def __init__(self, role, members):
        self.role = role; self.members = list(members)

class _Policy:
    def __init__(self, bindings=None, etag=b''):
        self.bindings = list(bindings or []); self.etag = etag

fake_iam_policy_pb2 = _types.SimpleNamespace(
    GetIamPolicyRequest=lambda **kw: _types.SimpleNamespace(**kw),
    SetIamPolicyRequest=lambda **kw: _types.SimpleNamespace(**kw),
)
fake_policy_pb2 = _types.SimpleNamespace(Binding=_Binding, Policy=_Policy)

existing = _Binding('roles/run.invoker', ['serviceAccount:other@p.iam.gserviceaccount.com'])
other = _Binding('roles/viewer', ['user:auditor@x.com'])
starting_policy = _Policy(bindings=[existing, other], etag=b'e')

captured = {}
fake_client = MagicMock()
fake_client.get_iam_policy.return_value = starting_policy
def set_iam_policy(request):
    captured['policy'] = request.policy
    return request.policy
fake_client.set_iam_policy.side_effect = set_iam_policy

with patch.dict(_sys.modules, {
    'google.iam.v1': _types.ModuleType('google.iam.v1'),
    'google.iam.v1.iam_policy_pb2': fake_iam_policy_pb2,
    'google.iam.v1.policy_pb2': fake_policy_pb2,
}):
    _grant_cloudrun_invoker_public(fake_client, 'projects/p/locations/r/services/s')

new_policy = captured['policy']
roles = {b.role for b in new_policy.bindings}
assert 'roles/viewer' in roles
assert 'roles/run.invoker' in roles
invoker = next(b for b in new_policy.bindings if b.role == 'roles/run.invoker')
assert 'allUsers' in invoker.members
assert 'serviceAccount:other@p.iam.gserviceaccount.com' in invoker.members
assert new_policy.etag == b'e'
